# 06 Forecasting and repeat purchase (BQ6): Python version

**BQ6:** What will weekly revenue be over the next quarter, and which customers will buy again?

- **Part A, weekly revenue forecast.** Hold out the last 13 weeks (Sep–Nov 2011), compare five models against the
  actuals, then refit the best model on all data and forecast the next 13 weeks (Dec 2011 – Feb 2012).
- **Part B, repeat purchase.** Predict which customers buy again within 90 days of a cut-off date
  (logistic regression, a decision tree and gradient boosting), trained on 2010 and tested on 2011 (out of time).

Same data, models and definitions as `06_forecast_repeat_r.ipynb`. The benchmarks, the regression and the logistic
model must match exactly; the ARIMA-error model should agree to about three decimals.

## Setup and data

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.tree import DecisionTreeClassifier, export_text
from statsmodels.tsa.statespace.sarimax import SARIMAX
import logging
import xgboost as xgb
from prophet import Prophet

logging.getLogger("cmdstanpy").disabled = True     # silence Stan progress messages
logging.getLogger("prophet").setLevel(logging.ERROR)

warnings.filterwarnings("ignore", module="statsmodels")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
DASH = ROOT / "reports" / "dashboard_data"
FIG.mkdir(parents=True, exist_ok=True)
DASH.mkdir(parents=True, exist_ok=True)

H = 13                    # forecast horizon and test length: one quarter of weeks
PERIOD = 365.25 / 7       # weeks per year, for the Fourier terms
K_MAX = 10

sales = pd.read_parquet(PROC / "sales_clean.parquet")
cancels = pd.read_parquet(PROC / "cancellations.parquet")
prod = sales.loc[sales["line_type"] == "product"].copy()      # T6
canc = cancels.loc[cancels["line_type"] == "product"].copy()  # T5
del sales, cancels
print(f"Product sales lines: {len(prod):,}   product cancellation lines: {len(canc):,}")

# Part A: Weekly revenue forecast

Weekly **net revenue** (gross product sales minus product cancellations, £ thousand) on a complete calendar,
as in BQ1–BQ2: closed weeks are 0, and the partial first and last weeks are dropped (104 complete weeks).

The two Christmas weeks behave very differently from the rest of the year (the business closes), so two dummy
variables mark them: `xmas_partial` (week starting 18–24 Dec, open only part of the week) and `xmas_closed`
(week starting 25–31 Dec).

In [ ]:
gross_w = prod.groupby("week_start")["revenue"].sum()
canc_w = canc.groupby("week_start")["revenue"].sum()            # negative values
net_w = gross_w.add(canc_w, fill_value=0)
all_weeks = pd.date_range(net_w.index.min(), net_w.index.max(), freq="W-MON")
y_all = (net_w.reindex(all_weeks, fill_value=0) / 1000).iloc[1:-1]   # £k; drop partial weeks
y_all.index.name = "week"
y_all.name = "net_revenue_k"

def design(weeks, t, K):
    """Intercept, linear trend, Christmas dummies and K Fourier pairs."""
    X = pd.DataFrame({"const": 1.0, "t": t.astype(float)}, index=weeks)
    X["xmas_partial"] = ((weeks.month == 12) & (weeks.day >= 18) & (weeks.day <= 24)).astype(float)
    X["xmas_closed"] = ((weeks.month == 12) & (weeks.day >= 25)).astype(float)
    for k in range(1, K + 1):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * t / PERIOD)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * t / PERIOD)
    return X

t_all = np.arange(1, len(y_all) + 1)
y_train, y_test = y_all.iloc[:-H], y_all.iloc[-H:]
t_train, t_test = t_all[:-H], t_all[-H:]
print(f"Weeks: {len(y_all)} ({y_all.index[0].date()} to {y_all.index[-1].date()})")
print(f"Train: {len(y_train)} weeks to {y_train.index[-1].date()}   Test: {len(y_test)} weeks from {y_test.index[0].date()}")
print(f"Christmas weeks in train: {design(y_train.index, t_train, 0)[['xmas_partial', 'xmas_closed']].sum().to_dict()}")

## A1. Number of Fourier terms

Seasonality is modelled with K pairs of sine and cosine waves of period one year: more pairs give a more detailed
seasonal shape but more parameters. With fewer than two years of training data, K is chosen by AIC on the training
weeks, computed in the same way in Python and R: `AIC = n·log(RSS/n) + 2p`.

In [ ]:
def aic_ols(y, X):
    fit = sm.OLS(y.values, X.values).fit()
    n, p = X.shape
    return n * np.log(fit.ssr / n) + 2 * p

aic = pd.Series({K: aic_ols(y_train, design(y_train.index, t_train, K)) for K in range(1, K_MAX + 1)},
                name="AIC").rename_axis("K")
K_BEST = int(aic.idxmin())
print(aic.round(2).to_string())
print(f"\nChosen K = {K_BEST}")

## A2. Models and test-set accuracy

| Model | Forecast for each test week |
|---|---|
| **Naive** | The last training week, repeated |
| **Seasonal naive** | The same week one year earlier (52 weeks back): the benchmark every model must beat |
| **Fourier regression** | OLS on trend, Christmas dummies and K Fourier pairs |
| **Fourier + AR(1) errors** | The same regression with first-order autoregressive errors (dynamic harmonic regression), fitted by maximum likelihood with `SARIMAX` |
| **Prophet** | Meta's additive model: piecewise-linear trend with automatic changepoints, yearly seasonality (10 Fourier pairs) and the two Christmas dummies as extra regressors |

**Not used, and why.** *ETS* with a 52-week season needs more yearly cycles than the two available: R's `ets()`
refuses seasonal periods above 24, and decomposing first (STL, then ETS on the adjusted series) fails too, because
STL needs more than two full cycles. *Seasonal ARIMA* with a 52-week seasonal lag is unstable on 91 weeks.
The Fourier terms (and Prophet, which uses them internally) describe the yearly shape with few parameters instead.

Accuracy: **MAE** (average absolute error, £k), **RMSE**, **MAPE** (average % error), **WAPE** (total absolute
error ÷ total actual revenue) and **bias** (total forecast ÷ total actual − 1).

In [ ]:
X_train = design(y_train.index, t_train, K_BEST)
X_test = design(y_test.index, t_test, K_BEST)

fc = pd.DataFrame({"actual": y_test})
fc["naive"] = y_train.iloc[-1]
fc["seasonal_naive"] = y_all.shift(52).loc[y_test.index]

ols = sm.OLS(y_train.values, X_train.values).fit()
fc["fourier_ols"] = ols.predict(X_test.values)

arx = SARIMAX(y_train.values, exog=X_train.values, order=(1, 0, 0), trend="n").fit(disp=False)
pred = arx.get_forecast(steps=H, exog=X_test.values).summary_frame(alpha=0.05)
fc["fourier_ar1"] = pred["mean"].values
fc["ar1_lo95"], fc["ar1_hi95"] = pred["mean_ci_lower"].values, pred["mean_ci_upper"].values

def prophet_forecast(weeks_fit, y_fit, weeks_new):
    """Fit Prophet on weekly data with Christmas regressors; return point forecasts for weeks_new."""
    def frame(w):
        d = design(w, np.zeros(len(w)), 0)
        return pd.DataFrame({"ds": w, "xmas_partial": d["xmas_partial"].values, "xmas_closed": d["xmas_closed"].values})
    m = Prophet(yearly_seasonality=10, weekly_seasonality=False, daily_seasonality=False,
                interval_width=0.95, uncertainty_samples=0)
    m.add_regressor("xmas_partial", standardize=False)
    m.add_regressor("xmas_closed", standardize=False)
    m.fit(frame(weeks_fit).assign(y=np.asarray(y_fit)))
    return m.predict(frame(weeks_new))["yhat"].values

fc["prophet"] = prophet_forecast(y_train.index, y_train.values, y_test.index)

def accuracy(actual, f):
    e = actual - f
    return {"MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()), "MAPE": (e.abs() / actual).mean() * 100,
            "WAPE": e.abs().sum() / actual.sum() * 100, "bias_pct": (f.sum() / actual.sum() - 1) * 100}

MODELS = ["naive", "seasonal_naive", "fourier_ols", "fourier_ar1", "prophet"]
acc = pd.DataFrame({m: accuracy(fc["actual"], fc[m]) for m in MODELS}).T
acc.index.name = "model"
print(f"AR(1) coefficient: {arx.params[X_train.shape[1]]:.4f}   "
      f"Fourier regression R² (train): {ols.rsquared:.3f}\n")
acc

In [ ]:
print("Test weeks, £k:")
print(fc[["actual"] + MODELS].round(1).to_string())

fig, ax = plt.subplots(figsize=(11, 4))
hist = y_all.loc[y_all.index >= y_test.index[0] - pd.Timedelta(weeks=52)]
ax.plot(hist.index, hist.values, color="grey", lw=1, label="Actual")
ax.plot(fc.index, fc["seasonal_naive"], ls="--", color="#81C784", label="Seasonal naive")
ax.plot(fc.index, fc["fourier_ols"], color="#64B5F6", label="Fourier regression")
ax.plot(fc.index, fc["fourier_ar1"], color="#FF8A65", label="Fourier + AR(1)")
ax.plot(fc.index, fc["prophet"], color="#BA68C8", label="Prophet")
ax.fill_between(fc.index, fc["ar1_lo95"], fc["ar1_hi95"], color="#FF8A65", alpha=0.15, label="95% interval")
ax.axvline(y_test.index[0], color="grey", ls=":", lw=1)
ax.set_ylabel("Net revenue (£k per week)"); ax.set_title("Test quarter: forecasts vs actual (Sep–Nov 2011)")
ax.legend(ncol=3, fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "bq6_forecast_test.png", dpi=150); plt.show()

## A3. Forecast for the next quarter

The Fourier + AR(1) model is refitted on all 104 weeks (K chosen again by AIC) and projected 13 weeks ahead.
The seasonal naive forecast is shown for comparison.

In [ ]:
aic_full = pd.Series({K: aic_ols(y_all, design(y_all.index, t_all, K)) for K in range(1, K_MAX + 1)})
K_FULL = int(aic_full.idxmin())
X_all = design(y_all.index, t_all, K_FULL)
fut_weeks = pd.date_range(y_all.index[-1] + pd.Timedelta(weeks=1), periods=H, freq="W-MON")
t_fut = np.arange(len(y_all) + 1, len(y_all) + H + 1)
X_fut = design(fut_weeks, t_fut, K_FULL)

arx_full = SARIMAX(y_all.values, exog=X_all.values, order=(1, 0, 0), trend="n").fit(disp=False)
pf = arx_full.get_forecast(steps=H, exog=X_fut.values).summary_frame(alpha=0.05)
future = pd.DataFrame({"forecast": pf["mean"].values, "lo95": pf["mean_ci_lower"].values,
                       "hi95": pf["mean_ci_upper"].values,
                       "seasonal_naive": y_all.iloc[-52:].iloc[:H].values,
                       "prophet": prophet_forecast(y_all.index, y_all.values, fut_weeks)}, index=fut_weeks)
future.index.name = "week"
print(f"K = {K_FULL}   AR(1) coefficient: {arx_full.params[X_all.shape[1]]:.4f}")
print(f"Next-quarter total: £{future['forecast'].sum():,.0f}k "
      f"(seasonal naive £{future['seasonal_naive'].sum():,.0f}k, Prophet £{future['prophet'].sum():,.0f}k)\n")
future.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_all.index, y_all.values, color="grey", lw=1, label="Actual")
ax.plot(future.index, future["forecast"], color="#FF8A65", label="Forecast (Fourier + AR(1))")
ax.fill_between(future.index, future["lo95"], future["hi95"], color="#FF8A65", alpha=0.15, label="95% interval")
ax.plot(future.index, future["seasonal_naive"], ls="--", color="#81C784", label="Seasonal naive")
ax.plot(future.index, future["prophet"], color="#BA68C8", label="Prophet")
ax.set_ylabel("Net revenue (£k per week)"); ax.set_title("Weekly net revenue and next-quarter forecast")
ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "bq6_forecast_future.png", dpi=150); plt.show()

# Part B: Repeat purchase

**Target:** a customer **buys again within 90 days** after a cut-off date (problem statement definition):
at least one product line not later cancelled (`cancelled_exact` = False).

| Set | Features measured | Outcome window |
|---|---|---|
| Train | 1 Dec 2009 – 31 Aug 2010 | 1 Sep – 29 Nov 2010 |
| Test | 1 Dec 2010 – 31 Aug 2011 | 1 Sep – 29 Nov 2011 |

Both sets use nine-month feature windows and the same season for the outcome, so the test is a fair out-of-time check.
Customers are those who bought in the feature window with positive net revenue.

| Feature | Definition (feature window) |
|---|---|
| `recency` | log(1 + days from the last purchase to the cut-off) |
| `frequency` | log(number of distinct purchase days) |
| `order_value` | log(average value of kept orders) |
| `variety` | log(number of distinct products) |
| `tenure` | days from the first purchase in the window to the cut-off |
| `cancel_share` | cancelled value ÷ gross sales |
| `is_uk` | 1 if at least half of the customer's lines are from the UK |

Continuous features are standardised with the **training** mean and standard deviation, so odds ratios are per one SD.

In [ ]:
def customer_set(start, cutoff):
    start, cutoff = pd.Timestamp(start), pd.Timestamp(cutoff)
    w = prod.loc[prod["has_customer"] & (prod["invoice_date"] >= start) & (prod["invoice_date"] < cutoff)]
    c = canc.loc[canc["customer_id"].notna() & (canc["invoice_date"] >= start) & (canc["invoice_date"] < cutoff)]
    f = w.groupby("customer_id").agg(first_buy=("date", "min"), last_buy=("date", "max"),
                                     frequency=("date", "nunique"), variety=("stock_code", "nunique"),
                                     gross=("revenue", "sum"), uk_share=("is_uk", "mean"))
    kept = w.loc[~w["cancelled_exact"]].groupby("customer_id").agg(kept_sales=("revenue", "sum"),
                                                                    orders=("invoice", "nunique"))
    f = f.join(kept)
    f["cancelled"] = (-c.groupby("customer_id")["revenue"].sum()).reindex(f.index).fillna(0)
    f["monetary"] = (f["gross"] - f["cancelled"]).round(2)
    f["avg_order_value"] = f["kept_sales"] / f["orders"]
    f["recency_days"] = (cutoff - f["last_buy"]).dt.days
    f["tenure_days"] = (cutoff - f["first_buy"]).dt.days
    f["cancel_share"] = (f["cancelled"] / f["gross"]).clip(upper=1)
    f["is_uk"] = (f["uk_share"] >= 0.5).astype(int)
    f = f.loc[(f["monetary"] > 0) & f["avg_order_value"].notna()].sort_index()

    out_end = cutoff + pd.Timedelta(days=90)
    nxt = prod.loc[prod["has_customer"] & ~prod["cancelled_exact"]
                   & (prod["invoice_date"] >= cutoff) & (prod["invoice_date"] < out_end), "customer_id"]
    f["bought_again"] = f.index.isin(nxt.unique()).astype(int)
    return f

train_c = customer_set("2009-12-01", "2010-09-01")
test_c = customer_set("2010-12-01", "2011-09-01")
for name, d in [("Train", train_c), ("Test ", test_c)]:
    print(f"{name}: {len(d):,} customers, bought again within 90 days: {d['bought_again'].mean():.1%}")

In [ ]:
CONT = ["recency", "frequency", "order_value", "variety", "tenure", "cancel_share"]
FEATURES = CONT + ["is_uk"]

def raw_features(f):
    return pd.DataFrame({"recency": np.log1p(f["recency_days"]), "frequency": np.log(f["frequency"]),
                         "order_value": np.log(f["avg_order_value"]), "variety": np.log(f["variety"]),
                         "tenure": f["tenure_days"].astype(float), "cancel_share": f["cancel_share"],
                         "is_uk": f["is_uk"]}, index=f.index)

raw_tr, raw_te = raw_features(train_c), raw_features(test_c)
mu, sd = raw_tr[CONT].mean(), raw_tr[CONT].std()          # training statistics (sample SD)
std_tr, std_te = raw_tr.copy(), raw_te.copy()
std_tr[CONT] = (raw_tr[CONT] - mu) / sd
std_te[CONT] = (raw_te[CONT] - mu) / sd

logit = sm.Logit(train_c["bought_again"], sm.add_constant(std_tr[FEATURES], has_constant="add")).fit(disp=0)
ci = np.exp(logit.conf_int())
odds = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": ci[0], "ci_high": ci[1],
                     "p_value": logit.pvalues}).drop("const")
odds.index.name = "feature"
print(f"Logistic regression (train): pseudo R² {logit.prsquared:.3f}   log-likelihood {logit.llf:,.2f}\n")
odds

## B2. Test performance and decision tree

**AUC / Gini / KS** measure how well the scores rank buyers above non-buyers; the **Brier score** is the mean squared
error of the predicted probabilities (lower is better; it also checks calibration). The tree uses depth 3, at least
100 customers per leaf and balanced classes (R: equal priors and `cp = -1`, as in BQ3).

**Gradient boosting** (XGBoost, the same library in Python and R) builds 300 shallow trees (depth 3, learning rate 0.05),
each correcting the errors of the previous ones. It can capture non-linear effects and interactions that logistic
regression cannot. Settings are fixed in advance, not tuned on the test year; `tree_method="exact"`, no row or column
sampling and one thread make the result deterministic, so Python and R should agree closely.

In [ ]:
def performance(y, score):
    auc = roc_auc_score(y, score)
    fpr, tpr, _ = roc_curve(y, score)
    return {"auc": auc, "gini": 2 * auc - 1, "ks": np.max(tpr - fpr), "brier": np.mean((score - y) ** 2)}

y_te = test_c["bought_again"].values
p_logit = logit.predict(sm.add_constant(std_te[FEATURES], has_constant="add")).values

tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, class_weight="balanced", random_state=42)
tree.fit(raw_tr[FEATURES], train_c["bought_again"])
p_tree = tree.predict_proba(raw_te[FEATURES])[:, 1]

XGB_PARAMS = {"objective": "binary:logistic", "max_depth": 3, "eta": 0.05, "min_child_weight": 1,
              "subsample": 1.0, "colsample_bytree": 1.0, "tree_method": "exact", "nthread": 1, "seed": 42}
XGB_ROUNDS = 300
dtrain = xgb.DMatrix(raw_tr[FEATURES].values, label=train_c["bought_again"].values, feature_names=FEATURES)
dtest = xgb.DMatrix(raw_te[FEATURES].values, feature_names=FEATURES)
gbm = xgb.train(XGB_PARAMS, dtrain, num_boost_round=XGB_ROUNDS)
p_gbm = gbm.predict(dtest)

perf = pd.DataFrame({"Logistic regression": performance(y_te, p_logit),
                     "Decision tree": performance(y_te, p_tree),
                     "Gradient boosting": performance(y_te, p_gbm)}).T
perf.index.name = "model"
print(f"Test base rate: {y_te.mean():.1%}   (Brier score of always predicting it: "
      f"{np.mean((y_te.mean() - y_te) ** 2):.3f})\n")
print(perf.round(3).to_string())
print("\nDecision tree (features on their raw log scale):")
print(export_text(tree, feature_names=FEATURES, decimals=2))
gain = pd.Series(gbm.get_score(importance_type="total_gain")).reindex(FEATURES).fillna(0)
print("Gradient boosting, share of total gain by feature:")
print((gain / gain.sum()).sort_values(ascending=False).round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for name, s, col in [("Logistic regression", p_logit, "#64B5F6"), ("Decision tree", p_tree, "#FF8A65"),
                     ("Gradient boosting", p_gbm, "#81C784")]:
    fpr, tpr, _ = roc_curve(y_te, s)
    ax.plot(fpr, tpr, color=col, label=f"{name} (AUC {roc_auc_score(y_te, s):.3f})")
ax.plot([0, 1], [0, 1], ls="--", color="grey")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Repeat purchase, 2011 test set"); ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(FIG / "bq6_repeat_roc.png", dpi=150); plt.show()

## B3. Gains table: who to target

Test customers are ranked by the logistic score and split into ten equal groups (decile 1 = highest score; ties
broken by customer ID). For a retention campaign, the table shows how many actual repeat buyers each decile holds.

In [ ]:
g = pd.DataFrame({"score": p_logit, "y": y_te}, index=test_c.index)
g["decile"] = np.ceil(10 * g["score"].rank(method="first", ascending=False) / len(g)).astype(int)
gains = g.groupby("decile").agg(customers=("y", "size"), buyers=("y", "sum"),
                                mean_score=("score", "mean"))
gains["buy_rate"] = gains["buyers"] / gains["customers"]
gains["cum_share_of_buyers"] = gains["buyers"].cumsum() / gains["buyers"].sum()
gains["lift"] = gains["buy_rate"] / y_te.mean()

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(gains.index, gains["buy_rate"] * 100, color="#64B5F6", label="Actual buy rate")
ax.plot(gains.index, gains["mean_score"] * 100, color="#FF8A65", marker="o", label="Mean predicted probability")
ax.axhline(y_te.mean() * 100, color="grey", ls="--", lw=1, label="Average")
ax.set_xlabel("Decile (1 = highest score)"); ax.set_ylabel("%"); ax.set_xticks(gains.index)
ax.set_title("Repeat purchase by score decile (2011 test)"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "bq6_repeat_gains.png", dpi=150); plt.show()
gains

## Export for the dashboard

In [ ]:
weekly_out = pd.DataFrame({"net_revenue_k": y_all})
weekly_out["set"] = np.where(weekly_out.index.isin(y_test.index), "test", "train")
weekly_out = weekly_out.join(fc[MODELS + ["ar1_lo95", "ar1_hi95"]])
weekly_out.round(4).to_csv(DASH / "forecast_weekly.csv")
acc.round(4).to_csv(DASH / "forecast_accuracy.csv")
future.round(4).to_csv(DASH / "forecast_next_quarter.csv")
odds.round(4).to_csv(DASH / "repeat_odds.csv")
perf.round(4).to_csv(DASH / "repeat_performance.csv")
gains.round(4).to_csv(DASH / "repeat_gains.csv")
for f in ["forecast_weekly.csv", "forecast_accuracy.csv", "forecast_next_quarter.csv",
          "repeat_odds.csv", "repeat_performance.csv", "repeat_gains.csv"]:
    print(f"{f:<28} {len(pd.read_csv(DASH / f)):>4} rows")